# Phase 7: Multi-Regime Backtesting

Five historical BTC windows, **regime definitions committed before running any backtest**.

**Critical labeling convention used throughout this notebook:**
- 🟡 **IN-SAMPLE (IS):** the pruned model was trained on this data.  
  Good performance here is expected from memorisation — it is **NOT evidence of skill**.
- 🟢 **OUT-OF-SAMPLE (OOS):** the model never saw this data.  
  This is the **only window where results carry meaning**.

Training period: `2020-01-01 → 2024-08-16`  
Val period (early stopping): `2024-08-18 → 2025-08-14`  
Test period (OOS): `2025-08-16 → 2026-08-11`

In [ ]:
from __future__ import annotations

import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt

from src.backtest.multi_regime import (
    REGIMES,
    format_regime_block,
    format_summary_table,
    run_multi_regime,
)
from src.backtest.report import load_backtest_config
from src.models.train import load_modeling_config

cfg    = load_modeling_config('../configs/config.yaml')
bt_cfg = load_backtest_config('../configs/config.yaml')
print('Config loaded.')
print(f"Symbol: {cfg['symbol']}  |  Threshold: {cfg['modeling']['confidence_threshold']}")

## 1. Pre-committed regime windows

These windows were defined in `src/backtest/multi_regime.py` **before** any backtest was run.  
Rationale for each window is embedded in the code.

In [ ]:
print(f"{'Name':<20} {'IS/OOS':<8} {'Start':<14} {'End':<14}")
print('-' * 58)
for r in REGIMES:
    tag = 'IS (trained)' if r.in_sample else 'OOS (held-out)'
    print(f"{r.name:<20} {tag:<14} {r.start:<14} {r.end}")

## 2. Run multi-regime backtests

In [ ]:
summaries = run_multi_regime('1d', cfg, bt_cfg, model_variant='pruned')
print(f"\nCompleted {len(summaries)} regime backtests.")

## 3. Equity curves — small multiples

🟡 = IN-SAMPLE (memorised)   🟢 = OUT-OF-SAMPLE (meaningful)

In [ ]:
IS_COLOR_LR  = '#e67e22'   # orange for IS LR
IS_COLOR_LGB = '#c0392b'   # red for IS LGB
IS_COLOR_BAH = '#95a5a6'   # grey for IS buy-and-hold
OOS_COLOR_LR  = '#2980b9'  # blue for OOS LR
OOS_COLOR_LGB = '#27ae60'  # green for OOS LGB
OOS_COLOR_BAH = '#7f8c8d'  # dark grey for OOS buy-and-hold

fig, axes = plt.subplots(1, 5, figsize=(22, 5), sharey=False)
fig.suptitle(
    'Phase 7: Multi-Regime Equity Curves  |  Pruned model (BTCUSDT 1d)\n'
    '🟡 IN-SAMPLE (NOT evidence of skill)   🟢 OUT-OF-SAMPLE (the only result that counts)',
    fontsize=11, y=1.02
)

for ax, s in zip(axes, summaries, strict=True):
    r = s.regime
    is_oos = 'IN-SAMPLE' if r.in_sample else 'OOS'
    border_color = '#e67e22' if r.in_sample else '#27ae60'
    lr_c  = IS_COLOR_LR  if r.in_sample else OOS_COLOR_LR
    lgb_c = IS_COLOR_LGB if r.in_sample else OOS_COLOR_LGB
    bah_c = IS_COLOR_BAH if r.in_sample else OOS_COLOR_BAH

    # Normalise equity curves to 100
    for eq, label, color, lw, ls in [
        (s.lr.equity_curve,      'LR',          lr_c,  1.8, '-'),
        (s.lgb.equity_curve,     'LGB',         lgb_c, 1.8, '--'),
        (s.buyhold.equity_curve, 'buy-and-hold', bah_c, 1.2, ':'),
    ]:
        normed = eq / eq.iloc[0] * 100
        ax.plot(normed.index, normed.values, label=label, color=color,
                linewidth=lw, linestyle=ls)

    ax.axhline(100, color='black', linewidth=0.5, linestyle='-', alpha=0.4)
    ax.set_title(
        f"[{is_oos}]\n{r.label}",
        fontsize=8, color=border_color, fontweight='bold'
    )
    ax.set_ylabel('Equity (start = 100)')
    ax.tick_params(axis='x', labelrotation=30, labelsize=7)

    # Add return summary in corner
    lr_ret  = s.lr.total_return_pct
    bah_ret = s.buyhold.total_return_pct
    ax.text(
        0.03, 0.03,
        f"LR: {lr_ret:+.1f}%\nB&H: {bah_ret:+.1f}%",
        transform=ax.transAxes, fontsize=7, verticalalignment='bottom',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7)
    )

    # Box colour codes IS vs OOS
    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(2)

axes[0].legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.savefig('../notebooks/07_equity_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: notebooks/07_equity_curves.png')

## 4. Detailed per-regime tables

Includes win rate, average trade P&L, and Sharpe.

In [ ]:
for s in summaries:
    print(format_regime_block(s))
    print()

## 5. Cross-regime summary table

In [ ]:
print(format_summary_table(summaries))

## 6. LGB signal analysis

LGB fires zero signals in every window — this section confirms that's structural.

In [ ]:
print('LGB signal audit across all regimes:')
print(f"{'Regime':<38} {'IS/OOS':<8} {'Days':>5} {'LGB trades':>11}")
print('-' * 65)
for s in summaries:
    tag = 'IS' if s.regime.in_sample else 'OOS'
    print(f"{s.regime.label:<38} {tag:<8} {s.n_usable_rows:>5} {s.lgb.n_trades:>11}")
print()
print(f"Confidence threshold: {cfg['modeling']['confidence_threshold']}")
print('LGB never reaches P(up) > 0.60 or P(up) < 0.40 in any of the 5 windows.')
print('This is a model characteristic, not a regime-specific observation.')

## 7. LR trade activity by regime

In [ ]:
import math

print('LR trade activity by regime:')
print(
    f"{'Regime':<38} {'IS/OOS':<8} {'Days':>5} {'Trades':>7}"
    f" {'Win rate':>9} {'Avg P&L':>8} {'LR ret':>8} {'B&H ret':>8}"
)
print('-' * 90)
for s in summaries:
    tag = 'IS' if s.regime.in_sample else 'OOS'
    wr = 'n/a' if math.isnan(s.lr.win_rate_pct) else f"{s.lr.win_rate_pct:.1f}%"
    avg_pnl = 'n/a' if math.isnan(s.lr.avg_trade_pct) else f"{s.lr.avg_trade_pct:+.2f}%"
    print(
        f"{s.regime.label:<38} {tag:<8} {s.n_usable_rows:>5} {s.lr.n_trades:>7}"
        f" {wr:>9} {avg_pnl:>8}"
        f" {s.lr.total_return_pct:>+7.1f}% {s.buyhold.total_return_pct:>+7.1f}%"
    )

## 8. Honest verdict

**The only result that matters is the OOS window (Phase 5 test, Aug 2025-Aug 2026).**

**OOS result:** LR returned **−46.8%** vs buy-and-hold **−46.0%** — essentially the
same outcome.  LR executed 70 round-trips, generating approximately 14 percentage
points of fee drag that it had to overcome just to match buy-and-hold.  It didn't.

**What the IS results reveal about memorisation (NOT skill):**

- *COVID crash (IS):* LR +116% vs B&H −6%.  The model was trained on this exact
  drawdown-and-recovery; of course it looks prescient — it memorised the pattern.
  
- *2022 bear (IS):* LR +14% vs B&H −65%.  Again, trained on this data.  The model
  knows the direction because it was trained on it.
  
- *2020-21 bull (IS):* LR +6.5% vs B&H +442%.  The same model that avoids bear
  drawdowns in IS misses most of the bull upside — even in-sample.  It's risk-averse,
  not directionally skilled.
  
- *2023 recovery (IS):* LR +17.8% vs B&H +165%.  Same pattern: protective in bears,
  absent in bulls.

**LGB fires no signals** at the 0.60 confidence threshold across all five windows,
IS or OOS.  It produces flat equity lines in every regime.  LGB's conservatism means
it avoids losses but also captures zero upside — it is not a usable signal generator
at this threshold.

**Conclusion:**  
Phase 7 confirms Phase 5's finding with historical context.  The model has no
demonstrated directional edge.  The IS bear-market results are memorisation artifacts,
not predictive ability.  The OOS result — the only honest signal — is flat vs
buy-and-hold after fees.  No model variant should be used for live trading.